# 🎓 TRAVER + McMiner: Misconception-Aware Coding Tutor
## Running on Google Colab Pro (T4/V100 GPU)

**Architecture:**
- **Tutor**: Llama-3.1-70B-Instruct via HuggingFace Inference API (0 GB GPU)
- **Student Simulator**: Mixtral-8x7B-Instruct via HuggingFace Inference API (0 GB GPU)
- **Verifier**: Pre-trained Verifier-7B loaded locally in 4-bit (~5 GB GPU)
- **McMiner**: Gemini API-based misconception detection (0 GB GPU)

**Phases:**
1. Run baseline TRAVER with pre-trained verifier
2. Run McMiner on student code to detect misconceptions
3. Inject misconceptions into tutor prompts and re-run
4. Evaluate: compare baseline vs misconception-aware TRAVER

**Prerequisites:** Add these keys in Colab → 🔑 Secrets (left sidebar):
- `HF_TOKEN` — your HuggingFace token
- `GOOGLE_API_KEY` — your Gemini API key

---
## 0. GPU Check & Configuration

In [ ]:
# Check GPU
!nvidia-smi

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===== LOAD API KEYS FROM COLAB SECRETS =====
# Go to the 🔑 icon in the left sidebar → add HF_TOKEN and GOOGLE_API_KEY
from google.colab import userdata
import os

HF_TOKEN = userdata.get('HF_TOKEN')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# ===== MODEL CONFIGURATION =====
TUTOR_MODEL_ID = "meta-llama/Llama-3.1-70B-Instruct"
STUDENT_MODEL_ID = "mistralai/Mixtral-8x7B-Instruct-v0.1"
HF_API_BASE_URL = "https://api-inference.huggingface.co/v1/"

# ===== STORAGE =====
DRIVE_DIR = "/content/drive/MyDrive/Coding-Tutor-Colab"
WORK_DIR = "/content/Coding-Tutor"
MODEL_DIR = f"{DRIVE_DIR}/models"
DATA_DIR = f"{DRIVE_DIR}/data"

# ===== PIPELINE SETTINGS =====
STUDENT_LEVELS = ["low_level", "med_level", "high_level"]
TUTOR_NUM_RESPONSES = 5  # Reduced from 10 for API rate limits

# ===== DATASET IDS =====
TUTOR_AGENTS_DATASET = "nlpscu/Tutor-Agents"
MCMINER_DATASET = "nlpscu/MCminer"

# Validate
assert HF_TOKEN, "Add HF_TOKEN to Colab Secrets (🔑 sidebar)!"
assert GOOGLE_API_KEY, "Add GOOGLE_API_KEY to Colab Secrets (🔑 sidebar)!"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print("✅ Configuration loaded from Colab Secrets.")

---
## 1. Install Dependencies & Clone Repos

In [ ]:
# Clone Coding-Tutor repo
if not os.path.exists("/content/Coding-Tutor"):
    !git clone https://github.com/iwangjian/Coding-Tutor.git /content/Coding-Tutor
else:
    print("✅ Coding-Tutor already exists.")

# Clone McMiner repo
if not os.path.exists("/content/mcminer"):
    !git clone https://github.com/taisazero/mcminer.git /content/mcminer
else:
    print("✅ mcminer already exists.")

In [ ]:
# Install dependencies
# NOTE: torch is pre-installed on Colab — do NOT reinstall it or pin a version.
# Pinning transformers==4.44.2 also hardcodes torch==2.4.0 which conflicts
# with Colab's torch 2.10.0+cu128. We let pip resolve versions freely.

# Batch 1: Core ML libraries (compatible with Colab's existing torch)
!pip install -q bitsandbytes peft accelerate safetensors

# Batch 2: Transformers + tokenizers (no torch version pinned)
!pip install -q "transformers>=4.44,<4.48" tiktoken sentencepiece protobuf

# Batch 3: API clients
!pip install -q openai huggingface_hub tenacity google-generativeai

# Batch 4: Utilities
!pip install -q tqdm python-dotenv

# Verify torch version is still the Colab-native one
import torch
print(f"✅ All dependencies installed.")
print(f"✅ torch version: {torch.__version__} (should be 2.10.x)")

---
## 2. Download Datasets & Pre-trained Models
Saved to Google Drive — no re-downloading on session restart.

In [ ]:
from huggingface_hub import snapshot_download, login
login(token=HF_TOKEN)

# --- Download Tutor-Agents dataset ---
tutor_data_dir = f"{DATA_DIR}/Tutor-Agents"
if not os.path.exists(tutor_data_dir):
    print("⬇️  Downloading Tutor-Agents dataset...")
    snapshot_download(
        TUTOR_AGENTS_DATASET, repo_type="dataset",
        local_dir=tutor_data_dir, token=HF_TOKEN)
    print("✅ Tutor-Agents dataset downloaded.")
else:
    print("✅ Tutor-Agents dataset already on Drive.")

# --- Download MCminer dataset ---
mcminer_data_dir = f"{DATA_DIR}/MCminer"
if not os.path.exists(mcminer_data_dir):
    print("⬇️  Downloading MCminer dataset...")
    snapshot_download(
        MCMINER_DATASET, repo_type="dataset",
        local_dir=mcminer_data_dir, token=HF_TOKEN)
    print("✅ MCminer dataset downloaded.")
else:
    print("✅ MCminer dataset already on Drive.")

print(f"\n📦 Dataset storage:")
!du -sh {DATA_DIR}/*

In [ ]:
# --- Download pre-trained Verifier-7B (5 shards, ~1.5 GB) ---
verifier_dir = f"{MODEL_DIR}/Verifier-7B"
if not os.path.exists(f"{verifier_dir}/part0"):
    print("⬇️  Downloading Verifier-7B checkpoints...")
    snapshot_download("jwanglvy/Verifier-7B", local_dir=verifier_dir, token=HF_TOKEN)
    print("✅ Verifier-7B downloaded.")
else:
    print("✅ Verifier-7B already on Drive.")

# --- Download Mistral-7B base model (verifier architecture) ---
mistral_dir = f"{MODEL_DIR}/Mistral-7B-v0.1"
if not os.path.exists(f"{mistral_dir}/config.json"):
    print("⬇️  Downloading Mistral-7B-v0.1...")
    snapshot_download("mistralai/Mistral-7B-v0.1", local_dir=mistral_dir, token=HF_TOKEN)
    print("✅ Mistral-7B-v0.1 downloaded.")
else:
    print("✅ Mistral-7B-v0.1 already on Drive.")

print(f"\n📦 Models:")
!du -sh {MODEL_DIR}/*

---
## 3. Set Up Working Directory

In [ ]:
import shutil

# Symlink output directory to Drive for persistence
output_link = f"{WORK_DIR}/output"
drive_output = f"{DRIVE_DIR}/output"
os.makedirs(drive_output, exist_ok=True)

if os.path.islink(output_link):
    os.unlink(output_link)
elif os.path.isdir(output_link):
    !cp -rn {output_link}/* {drive_output}/ 2>/dev/null; rm -rf {output_link}
os.symlink(drive_output, output_link)

# Copy MCminer data files into mcminer repo
for fname in ["misconception_bank.json", "problems_processed.json"]:
    src = f"{DATA_DIR}/MCminer/{fname}"
    dst = f"/content/mcminer/{fname}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)

# Copy corrupted_codes_best
src_dir = f"{DATA_DIR}/MCminer/corrupted_codes_best"
dst_dir = f"/content/mcminer/corrupted_codes_best"
if os.path.exists(src_dir) and not os.path.exists(dst_dir):
    shutil.copytree(src_dir, dst_dir)

print("✅ Working directories ready.")
print(f"  Output → {drive_output}")

---
## 4. Patch VLLMChat for HuggingFace Inference API

The original code expects local vLLM servers. We patch `VLLMChat` to:
- Use HF Inference API's OpenAI-compatible endpoint
- Handle `n > 1` by looping (HF doesn't support multi-completion)
- Add retry logic with exponential backoff

In [ ]:
import sys
sys.path.insert(0, WORK_DIR)
sys.path.insert(0, f"{WORK_DIR}/traver")

PATCH_FILE = f"{WORK_DIR}/traver/chatarena/backends/openai_vllm.py"

with open(PATCH_FILE, 'r') as f:
    original = f.read()

if "COLAB_PATCHED" not in original:
    patched = """# COLAB_PATCHED - HuggingFace Inference API compatible
from typing import List
import re, time
from openai import OpenAI
from .base import IntelligenceBackend
from ..message import Message, SYSTEM_NAME

END_OF_MESSAGE = "<EOS>"

class VLLMChat(IntelligenceBackend):
    stateful = False
    type_name = "vllm-chat"

    def __init__(self, vllm_api_key, vllm_endpoint, model_name_or_path,
                 temperature=0.75, top_p=0.95, max_tokens=500,
                 max_latest_messages=-1, **kwargs):
        super().__init__(model_name_or_path=model_name_or_path,
                         temperature=temperature, top_p=top_p,
                         max_tokens=max_tokens,
                         max_latest_messages=max_latest_messages, **kwargs)
        self.client = OpenAI(api_key=vllm_api_key, base_url=vllm_endpoint)
        self.model = model_name_or_path
        self.temperature = temperature
        self.top_p = top_p
        self.max_tokens = max_tokens
        self.max_latest_messages = max_latest_messages

    def _single_call(self, messages):
        for attempt in range(3):
            try:
                c = self.client.chat.completions.create(
                    model=self.model, messages=messages,
                    temperature=self.temperature, top_p=self.top_p,
                    max_tokens=self.max_tokens, n=1)
                r = c.choices[0].message.content
                return r.strip() if r else ""
            except Exception as e:
                wait = 5 * (attempt + 1)
                print(f"  API error (attempt {attempt+1}/3): {e}. Waiting {wait}s...")
                time.sleep(wait)
        return "[API Error]"

    def _get_response(self, messages, num_responses=1):
        if num_responses > 1:
            responses = []
            for i in range(num_responses):
                responses.append(self._single_call(messages))
                if i < num_responses - 1:
                    time.sleep(0.5)
            return responses
        return self._single_call(messages)

    def query(self, agent_name, role_desc, history_messages, global_prompt=None,
              request_msg=None, num_responses=1, *args, **kwargs):
        if global_prompt:
            system_prompt = f"{global_prompt.strip()}\n\nYour name: {agent_name}\n\nYour role: {role_desc}"
        else:
            system_prompt = role_desc

        if self.max_latest_messages > 0 and len(history_messages) > self.max_latest_messages:
            history_messages = history_messages[-self.max_latest_messages:]

        all_messages = [(SYSTEM_NAME, system_prompt)]
        for msg in history_messages:
            if msg.agent_name == SYSTEM_NAME:
                all_messages.append((SYSTEM_NAME, msg.content))
            else:
                all_messages.append((msg.agent_name, f"{msg.content}{END_OF_MESSAGE}"))

        if request_msg is not None:
            all_messages.append((SYSTEM_NAME, request_msg.content))
        else:
            all_messages.append((SYSTEM_NAME, f"Now you speak, {agent_name}.{END_OF_MESSAGE}"))

        messages = []
        for i, msg in enumerate(all_messages):
            if i == 0:
                assert msg[0] == SYSTEM_NAME
                messages.append({"role": "user", "content": msg[1]})
            elif i == len(all_messages) - 1:
                assert msg[0] == SYSTEM_NAME
                messages[-1]["content"] = f"{messages[-1]['content']}\n\n{msg[1]}"
            else:
                if msg[0] == agent_name:
                    messages.append({"role": "assistant", "content": msg[1]})
                else:
                    if messages[-1]["role"] == "user":
                        messages[-1]["content"] = f"{messages[-1]['content']}\n\n[{msg[0]}]: {msg[1]}"
                    else:
                        messages.append({"role": "user", "content": f"[{msg[0]}]: {msg[1]}"})

        response = self._get_response(messages, num_responses, *args, **kwargs)

        if num_responses > 1:
            response = [re.sub(r"^\s*\[.*]:", "", r).strip() for r in response]
            response = [re.sub(rf"^\s*{re.escape(agent_name)}\s*:", "", r).strip() for r in response]
            response = [re.sub(rf"{END_OF_MESSAGE}$", "", r).strip() for r in response]
            for idx, r in enumerate(response):
                if END_OF_MESSAGE in r:
                    response[idx] = r.split(END_OF_MESSAGE)[0].strip()
        else:
            response = re.sub(r"^\s*\[.*]:", "", response).strip()
            response = re.sub(rf"^\s*{re.escape(agent_name)}\s*:", "", response).strip()
            response = re.sub(rf"{END_OF_MESSAGE}$", "", response).strip()
            if END_OF_MESSAGE in response:
                response = response.split(END_OF_MESSAGE)[0].strip()
        return response
"""
    with open(PATCH_FILE, 'w') as f:
        f.write(patched)
    print("✅ VLLMChat patched for HF Inference API.")
else:
    print("✅ VLLMChat already patched.")

---
## 5. Phase 1: Run Baseline TRAVER

Runs the standard Traver pipeline with the pre-trained verifier.
Progress is checkpointed — safe to restart if session disconnects.

In [ ]:
for level in STUDENT_LEVELS:
    print(f"\n{'='*60}")
    print(f"🎓 Phase 1: TRAVER - {level}")
    print(f"{'='*60}")
    !cd {WORK_DIR} && python traver/run_traver.py \
        --tutor_setting traver \
        --namespace_file prompt/namespaces.json \
        --prompt_element_file prompt/prompt_elements_final.jsonl \
        --output_dir output/dialogue \
        --verifier_base_model_path {MODEL_DIR}/Mistral-7B-v0.1 \
        --verifier_model_dir {MODEL_DIR}/Verifier-7B \
        --tutor_model_name_or_path {TUTOR_MODEL_ID} \
        --tutor_num_responses {TUTOR_NUM_RESPONSES} \
        --student_model_name_or_path {STUDENT_MODEL_ID} \
        --student_setting {level} \
        --vllm_api_key {HF_TOKEN} \
        --vllm_endpoint_tutor {HF_API_BASE_URL} \
        --vllm_endpoint_student {HF_API_BASE_URL} \
        --show_description false \
        --show_message true

print("\n✅ Phase 1 complete! Dialogues → output/dialogue/traver/")

---
## 6. Phase 2: Run McMiner on Student Code

Extract student code from Phase 1 dialogues, then use McMiner (via Gemini API) to identify misconceptions.

In [ ]:
import json, glob, re

def extract_student_code(dialogue_dir):
    """Extract code snippets from student messages in TRAVER dialogues."""
    samples = []
    for f_path in glob.glob(f"{dialogue_dir}/**/simulated_dialogs.jsonl", recursive=True):
        print(f"📖 {f_path}")
        with open(f_path) as f:
            for line in f:
                data = json.loads(line.strip())
                ns = data.get("namespace", "")
                for turn_idx, turn in enumerate(data.get("conversation", [])):
                    if "student" not in turn:
                        continue
                    msg = turn["student"]
                    blocks = re.findall(r"```(?:python)?\s*(.*?)```", msg, re.DOTALL)
                    if not blocks and any(k in msg for k in ["def ", "class ", "import ", "return "]):
                        blocks = [msg]
                    for ci, code_str in enumerate(blocks):
                        code_str = code_str.strip()
                        if len(code_str) > 20:
                            samples.append({"namespace": ns, "turn_index": turn_idx,
                                          "code_index": ci, "student_code": code_str,
                                          "source_file": f_path})
    return samples

code_samples = extract_student_code(f"{DRIVE_DIR}/output/dialogue/traver")
code_path = f"{DRIVE_DIR}/output/mcminer/extracted_student_code.json"
os.makedirs(os.path.dirname(code_path), exist_ok=True)
with open(code_path, "w") as f:
    json.dump(code_samples, f, indent=2)
print(f"\n📊 Extracted {len(code_samples)} code samples")

In [ ]:
import time
import google.generativeai as genai

# Configure Gemini with key from Colab Secrets
genai.configure(api_key=GOOGLE_API_KEY)

MCMINER_PROMPT = """You are an expert programming instructor. Analyze this student code for
programming MISCONCEPTIONS (fundamental misunderstandings, NOT just bugs/typos).

Student's code:
```python
{student_code}
```

If you find a misconception, respond:
<misconception>
<description>Concise description of the misconception</description>
<explanation>What the student believes vs reality</explanation>
<confidence>high/medium/low</confidence>
</misconception>

If no misconception (code is correct or just has typos): <misconception>NONE</misconception>"""

def run_mcminer_gemini(samples):
    """Run McMiner misconception detection using Gemini API."""
    model = genai.GenerativeModel("gemini-2.0-flash")
    results = []

    for i, s in enumerate(samples):
        prompt = MCMINER_PROMPT.format(student_code=s["student_code"])
        try:
            resp = model.generate_content(prompt)
            raw = resp.text
        except Exception as e:
            print(f"  ⚠️ Sample {i}: {e}")
            raw = "<misconception>NONE</misconception>"
            time.sleep(2)

        desc = re.search(r"<description>(.*?)</description>", raw, re.DOTALL)
        expl = re.search(r"<explanation>(.*?)</explanation>", raw, re.DOTALL)
        conf = re.search(r"<confidence>(.*?)</confidence>", raw, re.DOTALL)
        is_none = "NONE" in raw and not desc

        results.append({
            **s,
            "misconception_detected": not is_none,
            "misconception_description": desc.group(1).strip() if desc else None,
            "misconception_explanation": expl.group(1).strip() if expl else None,
            "confidence": conf.group(1).strip() if conf else None,
            "raw_response": raw
        })

        if (i + 1) % 10 == 0:
            print(f"  Processed {i+1}/{len(samples)} samples...")

    return results

mcminer_results = run_mcminer_gemini(code_samples)
out_path = f"{DRIVE_DIR}/output/mcminer/misconception_results.json"
with open(out_path, "w") as f:
    json.dump(mcminer_results, f, indent=2)

detected = sum(1 for r in mcminer_results if r["misconception_detected"])
print(f"\n📊 McMiner: {detected}/{len(mcminer_results)} misconceptions ({100*detected/max(len(mcminer_results),1):.1f}%)")

---
## 7. Phase 3: Inject Misconceptions & Re-run TRAVER

Augment the tutor prompt with misconception information from McMiner.

In [ ]:
# Build misconception lookup: namespace -> misconceptions
misconception_lookup = {}
for r in mcminer_results:
    if r["misconception_detected"] and r["misconception_description"]:
        ns = r["namespace"]
        if ns not in misconception_lookup:
            misconception_lookup[ns] = []
        misconception_lookup[ns].append({
            "description": r["misconception_description"],
            "explanation": r["misconception_explanation"],
            "confidence": r["confidence"],
            "turn_index": r["turn_index"]})

lookup_path = f"{DRIVE_DIR}/output/mcminer/misconception_lookup.json"
with open(lookup_path, "w") as f:
    json.dump(misconception_lookup, f, indent=2)

print(f"📊 Lookup: {len(misconception_lookup)} namespaces, "
      f"{sum(len(v) for v in misconception_lookup.values())} misconceptions")

In [ ]:
# Create misconception-aware TRAVER wrapper script
script_content = f'''#!/usr/bin/env python3
"""Wrapper: injects McMiner misconceptions into tutor prompt."""
import json, sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "traver"))
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

with open("{lookup_path}", "r") as f:
    MISC_LOOKUP = json.load(f)

from traver.utils.make_prompt import prompt_tutor as orig_prompt_tutor
import traver.utils.make_prompt as mp

def mc_prompt_tutor(d, tokenizer, setting="base", max_code_context=1024):
    prompt = orig_prompt_tutor(d, tokenizer, setting=setting, max_code_context=max_code_context)
    ns = d.get("namespace", "")
    if ns in MISC_LOOKUP and setting == "base":
        txt = "\\n\\n--- STUDENT MISCONCEPTION ALERT ---\\n"
        txt += "Detected misconceptions in this student\'s code:\\n"
        for i, m in enumerate(MISC_LOOKUP[ns], 1):
            txt += f"  {{i}}. {{m[\'description\']}}\\n"
            if m.get("explanation"):
                txt += f"     Context: {{m[\'explanation\']}}\\n"
        txt += ("\\nGuide the student to discover and correct these misconceptions "
               "step-by-step using Socratic questioning. Do NOT give direct answers.\\n"
               "--- END ALERT ---")
        prompt += txt
    return prompt

mp.prompt_tutor = mc_prompt_tutor
from traver.run_traver import parse_args, main
args = parse_args()
print(f"\\n🧠 McMiner-TRAVER: {{len(MISC_LOOKUP)}} namespaces with misconceptions")
main(args)
\'\'\'

script_path = f"{WORK_DIR}/run_traver_mcminer.py"
with open(script_path, "w") as f:
    f.write(script_content)
print(f"✅ Created: {script_path}")

In [ ]:
for level in STUDENT_LEVELS:
    print(f"\n{'='*60}")
    print(f"🧠 Phase 3: McMiner-TRAVER - {level}")
    print(f"{'='*60}")
    !cd {WORK_DIR} && python run_traver_mcminer.py \
        --tutor_setting traver \
        --namespace_file prompt/namespaces.json \
        --prompt_element_file prompt/prompt_elements_final.jsonl \
        --output_dir output/dialogue_mcminer \
        --verifier_base_model_path {MODEL_DIR}/Mistral-7B-v0.1 \
        --verifier_model_dir {MODEL_DIR}/Verifier-7B \
        --tutor_model_name_or_path {TUTOR_MODEL_ID} \
        --tutor_num_responses {TUTOR_NUM_RESPONSES} \
        --student_model_name_or_path {STUDENT_MODEL_ID} \
        --student_setting {level} \
        --vllm_api_key {HF_TOKEN} \
        --vllm_endpoint_tutor {HF_API_BASE_URL} \
        --vllm_endpoint_student {HF_API_BASE_URL} \
        --show_description false \
        --show_message true

print("\n✅ Phase 3 complete!")

---
## 8. Phase 4: Evaluate & Compare

In [ ]:
def load_dialogues(d):
    all_d = []
    for fp in glob.glob(f"{d}/**/simulated_dialogs.jsonl", recursive=True):
        with open(fp) as f:
            for line in f:
                data = json.loads(line.strip())
                for p in fp.split("/"):
                    if p in ["low_level", "med_level", "high_level"]:
                        data["student_level"] = p
                        break
                all_d.append(data)
    return all_d

def show_stats(dialogues, label):
    levels = {}
    for d in dialogues:
        lv = d.get("student_level", "unknown")
        if lv not in levels:
            levels[lv] = {"count": 0, "turns": 0}
        levels[lv]["count"] += 1
        levels[lv]["turns"] += len(d.get("conversation", []))
    total = sum(l["turns"] for l in levels.values())
    n = max(len(dialogues), 1)
    print(f"\n📊 {label}:")
    print(f"  Total: {len(dialogues)} dialogues, {total/n:.1f} avg turns")
    for lv, ld in sorted(levels.items()):
        print(f"  {lv}: {ld['count']} dialogues, {ld['turns']/max(ld['count'],1):.1f} avg turns")

print("="*60)
print("📈 EVALUATION COMPARISON")
print("="*60)
baseline = load_dialogues(f"{DRIVE_DIR}/output/dialogue/traver")
mcminer = load_dialogues(f"{DRIVE_DIR}/output/dialogue_mcminer/traver")
show_stats(baseline, "Baseline TRAVER")
show_stats(mcminer, "McMiner-TRAVER")

In [ ]:
# Full evaluation instructions
print("⚠️  Full pass@k evaluation requires the EvoCodeBench execution environment.")
print("   The dialogue statistics above provide a quick comparison.")
print()
print("For full evaluation, run these scripts:")
print(f"  1. cd {WORK_DIR}")
print(f"  2. bash scripts/run/run_pretest.sh")
print(f"  3. bash scripts/run/run_code_gen.sh")
print(f"  4. bash scripts/run/run_coding_test.sh")
print(f"  5. python scripts/eval/eval_TOR.py")

---
## 📋 Summary

| Phase | Description | Output |
|-------|-------------|--------|
| **1** | Baseline TRAVER dialogues | `output/dialogue/traver/` |
| **2** | McMiner misconception analysis | `output/mcminer/` |
| **3** | McMiner-aware TRAVER dialogues | `output/dialogue_mcminer/traver/` |
| **4** | Evaluation comparison | Stats above |

All outputs saved to Google Drive: `/content/drive/MyDrive/Coding-Tutor-Colab/output/`

**Session-safe**: Progress is checkpointed. Re-run cells if disconnected.